# End-to-End CatBoost Workflow

This notebook is a fully worked example of how to use `portfolio_toolkit` with CatBoost gradient-boosted models.
It trains on **`shared_set_1`** — the full S&P 500 universe (503 tickers) — for maximum diversity and generalisation.

It covers the full workflow:

1. load a shared dataset
2. add built-in toolkit features
3. engineer a custom notebook-local feature
4. build forward return / alpha / volatility targets
5. split into train / validation / test using the shared split rules
6. train three CatBoost regressors (return, alpha, volatility)
7. emit a standardized prediction table
8. turn predictions into a `PortfolioWeights` object
9. run the shared backtest
10. write reports and artifacts
11. log everything to MLflow as run `Felix_Init`


## Running This Notebook In Colab

If you want to run this notebook in Google Colab, start by cloning the repo into the Colab session and installing the toolkit in editable mode.

Steps:

1. Set `REPO_URL` below to your GitHub repo URL.
2. Run the bootstrap cell once.
3. After that, the rest of the notebook can import `portfolio_toolkit` normally.

If you are running locally, the same cell will automatically fall back to the repository on your machine.


In [4]:
# Colab / local bootstrap cell
# - In Colab: clone the repo, install the package, and point repo_root at /content/...
# - Locally: just point repo_root at this repository on disk

import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/<your-user-or-org>/<your-repo>.git'  # replace with your real repo URL
    REPO_DIR = '/content/Portfolio-Optimizer'

    if '<your-user-or-org>' in REPO_URL or '<your-repo>' in REPO_URL:
        raise ValueError('Set REPO_URL to your real GitHub repository URL before running this cell in Colab.')

    !rm -rf "$REPO_DIR"
    !git clone "$REPO_URL" "$REPO_DIR"
    %cd "$REPO_DIR"
    %pip install -e ".[dev]"

    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()

print('repo_root =', repo_root)


repo_root = /Users/beizhang/Portfolio-Optimization-Lib


## What This Example Is Doing

This notebook uses `shared_set_2` by default instead of `shared_set_1`.

Why:

- `shared_set_1` is the full S&P 500 universe, so the first download is much larger.
- `shared_set_2` is smaller and faster for a first MLP example.

Once you understand the pattern, you can switch `dataset_name` to `shared_set_1` and run the exact same workflow over a much broader universe.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf

from portfolio_toolkit import (
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    log_predictions,
    make_forward_alpha_target,
    make_forward_realized_vol_target,
    make_forward_return_target,
    slice_split,
    split_dates,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_risk_adjusted,
    backtest_weights,
    write_backtest_artifacts,
)

print('Imports loaded successfully.')


/Users/beizhang/Portfolio-Optimization-Lib/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports loaded successfully.


In [3]:
# Resolve repo_root robustly — always anchor to this notebook's location.
repo_root = Path(__file__).resolve().parents[2] if '__file__' in dir() else Path().resolve()
# If the above still resolves incorrectly, uncomment and set explicitly:
# repo_root = Path('/Users/beizhang/Portfolio-Optimization-Lib')
print('repo_root:', repo_root)
dataset_name = 'shared_set_1'
model_name = 'catboost_sp500'
horizon = 5
output_dir = repo_root / 'runs' / 'catboost_sp500_workflow'
output_dir.mkdir(parents=True, exist_ok=True)

spec = get_dataset_spec(dataset_name, repo_root=repo_root)
splits = split_dates(dataset_name, repo_root=repo_root)

print('Dataset preset:', dataset_name)
print('Dataset display name:', spec.name)
print('Number of modeled tickers:', len(spec.tickers))
print('Benchmark ticker:', spec.benchmark_ticker)
print('Train/Val/Test windows:', splits)


Dataset preset: shared_set_2
Dataset display name: growth_tech_innovation
Number of modeled tickers: 26
Benchmark ticker: SPY
Train/Val/Test windows: {'train': (Timestamp('2014-01-02 00:00:00'), Timestamp('2019-12-31 00:00:00')), 'val': (Timestamp('2020-01-02 00:00:00'), Timestamp('2021-12-31 00:00:00')), 'test': (Timestamp('2022-01-03 00:00:00'), Timestamp('2025-12-31 00:00:00'))}


## 1. Load Shared Price Data

The toolkit's `load_prices(...)` function is the shared data entrypoint.

What it does:

- reads the selected dataset preset from `configs/datasets.toml`
- downloads daily OHLCV data with `yfinance` if it is not already cached
- always includes `SPY` as the benchmark series
- validates and normalizes the dataframe before returning it

This is one of the main standardization points in the repo: everyone starts from the same dataset preset and the same split boundaries.

In [4]:
prices = load_prices(dataset_name, repo_root=repo_root)

print('Price frame shape:', prices.shape)
print('Date range:', prices['date'].min(), '->', prices['date'].max())
print('Number of unique tickers in price frame:', prices['ticker'].nunique())
display(prices.head())

Price frame shape: (78468, 8)
Date range: 2014-01-02 00:00:00 -> 2025-12-31 00:00:00
Number of unique tickers in price frame: 26


,date,ticker,open,high,low,close,adj_close,volume
0,2014-01-02,AAPL,19.845715,19.893929,19.715000,19.754642,17.140659,234684800
1,2014-01-03,AAPL,19.745001,19.775000,19.301071,19.320715,16.764149,392467600
2,2014-01-06,AAPL,19.194643,19.528570,19.057142,19.426071,16.855572,412610800
3,2014-01-07,AAPL,19.440001,19.498571,19.211430,19.287144,16.735027,317209200
4,2014-01-08,AAPL,19.243214,19.484285,19.238930,19.409286,16.841009,258529600


## 2. Add Built-In Toolkit Features

We start with a moderate built-in feature set.

These are all created by the shared library, which means other teammates can use the same starting point if they want:

- momentum
- volatility
- RSI
- moving-average distance
- volume z-score
- benchmark-relative return
- intraday range and beta vs SPY

This is a good example of the intended workflow:

- shared features for consistency and speed
- custom notebook-local features on top when you want to experiment

In [5]:
# NOTE: base features trimmed to minimum for fast runtime on shared_set_1.
# Restore full list when running on a faster machine.
base_feature_names = [
    'return_5d',
    'return_20d',
    'vol_20d',
    'momentum_20d',
    'excess_return_20d_vs_spy',
]

base_features = build_features(prices, feature_names=base_feature_names)
print('Base feature frame shape:', base_features.shape)
display(base_features)


Base feature frame shape: (78468, 14)


,date,ticker,return_5d,return_20d,vol_20d,momentum_20d,momentum_60d,rsi_14,price_to_sma_20d,price_to_sma_50d,volume_zscore_20d,excess_return_20d_vs_spy,intraday_range,beta_20d_spy
0,2014-01-02,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.009058,NaN
1,2014-01-03,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.024530,NaN
2,2014-01-06,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.024268,NaN
3,2014-01-07,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.014888,NaN
4,2014-01-08,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.012641,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78463,2025-12-24,TXN,0.015130,0.094950,0.016532,0.094950,-0.027317,42.843447,0.000438,0.047161,-1.596433,0.069173,0.006888,1.661551
78464,2025-12-26,TXN,0.003916,0.069731,0.016080,0.069731,-0.010706,34.882494,-0.004217,0.045292,-0.964098,0.051090,0.011816,1.574360
78465,2025-12-29,TXN,-0.003403,0.044096,0.015884,0.044096,-0.027764,35.663544,-0.012978,0.038043,-0.696523,0.034595,0.014514,1.532279
78466,2025-12-30,TXN,-0.019014,0.043173,0.015893,0.043173,-0.018491,38.053565,-0.016500,0.036399,-0.727059,0.030281,0.006955,1.581910


## 3. Add New Custom Features In The Notebook

This is where developers keep their freedom. The toolkit supplies the base feature set;
everything below is notebook-local and experimental.

We add three categories of custom features:

**Stock-level signals:**
- `mom_vol_ratio` — momentum normalised by volatility
- `trend_spread` — short-term SMA minus long-term SMA
- `quality_signal` — excess return penalised by volatility
- `range_volume_interaction` — intraday range × volume z-score
- `rolling_sharpe_20d` — annualised 20-day rolling Sharpe ratio per ticker

**Volatility regime features:**
- `vol_of_vol` — std of rolling 5d vol over a 20-day window; captures whether vol is stable or erratic
- `drawdown_20d` — current price vs 20-day high; a stock at -25% behaves differently to one at ATH

**Macro / regime features (date-level, same value for all tickers on a given day):**
- `vix_close` — CBOE VIX, the market fear gauge
- `vix_change_5d` — 5-day change in VIX (rising = fear increasing)
- `tlt_momentum_20d` — 20-day return of TLT (Treasury bond ETF, inverse proxy for 10Y yield)
- `rate_regime` — sign of `tlt_momentum_20d`: +1 = rates falling, −1 = rates rising

All features are derived purely from price and volume data — no ticker identity is encoded.
This means the model generalises to any ticker or ETF it has not seen during training,
as long as OHLCV history is available.


In [6]:
frame = base_features.copy()

# NOTE: only instant arithmetic features kept for fast runtime.
# Macro features (VIX/TLT) and all rolling features temporarily removed.
eps = 1e-6

frame['mom_vol_ratio'] = frame['momentum_20d'] / (frame['vol_20d'].abs() + eps)
frame['quality_signal'] = frame['excess_return_20d_vs_spy'] - 0.5 * frame['vol_20d']

macro_feature_names = []

custom_feature_names = [
    'mom_vol_ratio',
    'quality_signal',
]

all_feature_names = base_feature_names + custom_feature_names
print(f'Total features: {len(all_feature_names)} ({len(base_feature_names)} base + {len(custom_feature_names)} custom)')
display(frame.loc[:, ['date', 'ticker'] + custom_feature_names].head())


,date,ticker,mom_vol_ratio,trend_spread,quality_signal,range_volume_interaction
0,2014-01-02,AAPL,NaN,NaN,NaN,NaN
1,2014-01-03,AAPL,NaN,NaN,NaN,NaN
2,2014-01-06,AAPL,NaN,NaN,NaN,NaN
3,2014-01-07,AAPL,NaN,NaN,NaN,NaN
4,2014-01-08,AAPL,NaN,NaN,NaN,NaN


## 4. Build Targets

We are going to fit three separate small MLPs with the exact same input features:

- one model for `expected_return`
- one model for `expected_alpha`
- one model for `expected_volatility`

This is not required, but it demonstrates the richer prediction contract supported by the toolkit.

Target builders used here:

- `make_forward_return_target(...)`
- `make_forward_alpha_target(...)`
- `make_forward_realized_vol_target(...)`

In [7]:
return_targets = make_forward_return_target(prices, horizon=horizon)
alpha_targets = make_forward_alpha_target(prices, horizon=horizon)
vol_targets = make_forward_realized_vol_target(prices, window=horizon)

target_frame = frame.merge(return_targets, on=['date', 'ticker'], how='left')
target_frame = target_frame.merge(alpha_targets, on=['date', 'ticker'], how='left')
target_frame = target_frame.merge(vol_targets, on=['date', 'ticker'], how='left')

# Drop rows only after all features and targets are assembled.
# This is the usual notebook pattern because long-window features and forward targets
# naturally create missing values near the beginning and end of each ticker history.
target_frame = target_frame.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

return_target_col = f'forward_return_{horizon}d'
alpha_target_col = f'forward_alpha_{horizon}d_vs_spy'
vol_target_col = f'forward_realized_vol_{horizon}d'

print('Modeling frame shape after dropping nulls:', target_frame.shape)
display(target_frame.head())

Modeling frame shape after dropping nulls: (76765, 21)


,date,ticker,return_5d,return_20d,vol_20d,momentum_20d,momentum_60d,rsi_14,price_to_sma_20d,price_to_sma_50d,...,excess_return_20d_vs_spy,intraday_range,beta_20d_spy,mom_vol_ratio,trend_spread,quality_signal,range_volume_interaction,forward_return_5d,forward_alpha_5d_vs_spy,forward_realized_vol_5d
0,2014-03-31,AAPL,-0.004544,0.017016,0.006838,0.017016,-0.023823,50.700538,0.006098,0.014112,...,0.001579,0.009092,0.415726,2.488208,-0.008015,-0.001839,-0.011534,-0.024724,-0.010446,0.146483
1,2014-04-01,AAPL,-0.006129,0.019595,0.006966,0.019595,0.007232,54.962510,0.014311,0.023226,...,0.011594,0.009416,0.468993,2.812653,-0.008915,0.008111,-0.005887,-0.033619,-0.016886,0.108591
2,2014-04-02,AAPL,0.005132,0.019141,0.006963,0.019141,0.003433,63.014063,0.015029,0.025053,...,0.008682,0.005935,0.465322,2.748570,-0.010024,0.005201,-0.005713,-0.022542,-0.013064,0.163991
3,2014-04-03,AAPL,0.002475,0.015148,0.007125,0.015148,0.003657,66.199215,0.007236,0.018312,...,0.008333,0.009020,0.501741,2.125688,-0.011076,0.004770,-0.011168,-0.028415,0.000584,0.172606
4,2014-04-04,AAPL,-0.009388,0.002602,0.007726,0.002602,-0.015561,55.243543,-0.005921,0.005940,...,0.008112,0.017713,0.631225,0.336719,-0.011861,0.004249,0.012090,-0.022959,0.003275,0.164265


## 5. Shared Train / Validation / Test Splits And Feature Scaling

This section does two very important things:

1. **Splits** the feature frame into train / val / test using the shared date windows
   defined in `datasets.toml` — so every model in the repo is evaluated on the same periods.

2. **Standardizes** features using *only* training-set statistics (mean and std),
   then applies those same statistics to val and test. This prevents data leakage.

3. **Saves the scaler** to disk as a JSON file so it can be reloaded at inference time
   and applied to any ticker — including ETFs not seen during training.
   Because all features are derived from price/volume behavior (not ticker identity),
   the scaler generalises to any OHLCV time series.


In [8]:
import json as _json

train = slice_split(target_frame, dataset_name, 'train', repo_root=repo_root)
val   = slice_split(target_frame, dataset_name, 'val',   repo_root=repo_root)
test  = slice_split(target_frame, dataset_name, 'test',  repo_root=repo_root)

print('Train rows:', len(train))
print('Val rows:  ', len(val))
print('Test rows: ', len(test))

# ── Build scaler from training data only (no leakage) ────────────────────────
train_means = train[all_feature_names].mean()
train_stds  = train[all_feature_names].std(ddof=0).replace(0.0, 1.0)

# Save scaler to disk so it can be reloaded for inference on unseen tickers / ETFs.
# At inference time: load this file, reconstruct means/stds, apply standardize().
# Ensure output directory exists before writing scaler.
output_dir.mkdir(parents=True, exist_ok=True)
scaler_path = output_dir / 'feature_scaler.json'
scaler_dict = {
    'feature_names': all_feature_names,
    'means':         train_means.to_dict(),
    'stds':          train_stds.to_dict(),
    'dataset':       dataset_name,
    'horizon':       horizon,
}
with open(scaler_path, 'w') as f:
    _json.dump(scaler_dict, f, indent=2)
print(f'Scaler saved to: {scaler_path}')

# ── Standardize splits ───────────────────────────────────────────────────────
def standardize(feature_frame: pd.DataFrame) -> np.ndarray:
    """Apply training-set scaler to any dataframe with matching feature columns.
    Works on any ticker universe — ETFs, single stocks, or mixed.
    """
    standardized = (feature_frame[all_feature_names] - train_means) / train_stds
    return standardized.fillna(0.0).to_numpy(dtype=float)

X_train = standardize(train)
X_val   = standardize(val)
X_test  = standardize(test)

y_train_return = train[return_target_col].to_numpy(dtype=float)
y_val_return   = val[return_target_col].to_numpy(dtype=float)
y_test_return  = test[return_target_col].to_numpy(dtype=float)

y_train_alpha  = train[alpha_target_col].to_numpy(dtype=float)
y_val_alpha    = val[alpha_target_col].to_numpy(dtype=float)

y_train_vol    = train[vol_target_col].to_numpy(dtype=float)
y_val_vol      = val[vol_target_col].to_numpy(dtype=float)

print('X_train shape:', X_train.shape)
print('Feature count:', X_train.shape[1])
print('NaN values after standardization:', np.isnan(X_train).sum())


Train rows: 37695
Val rows: 13130
Test rows: 25940
X_train shape: (37695, 16)
Feature count: 16


In [ ]:
# ── How to reload the scaler for inference on unseen tickers / ETFs ─────────
# This block is for reference — it shows how to apply the saved scaler
# to a completely new dataframe (e.g. ETF test data not in shared_set_2).
#
# import json as _json
# import numpy as np, pandas as pd
# with open("runs/catboost_end_to_end_workflow/feature_scaler.json") as f:
#     scaler = _json.load(f)
#
# saved_means = pd.Series(scaler["means"])
# saved_stds  = pd.Series(scaler["stds"])
# feature_names = scaler["feature_names"]
#
# def standardize_new(df: pd.DataFrame) -> np.ndarray:
#     return ((df[feature_names] - saved_means) / saved_stds).fillna(0.0).to_numpy(dtype=float)
#
# X_new = standardize_new(new_etf_feature_frame)
# preds = return_model.predict(X_new)  # works on any ticker

print('Scaler reload pattern shown above — no action needed in training run.')


## 6. Train CatBoost Models

We train three separate CatBoost regressors — one each for:

- **Expected return** (`forward_return_5d`)
- **Expected alpha** (`forward_alpha_5d_vs_spy`)
- **Expected volatility** (`forward_realized_vol_5d`)

CatBoost handles tabular data well out of the box: it is robust to feature scale,
handles moderate noise, and trains fast enough for notebook iteration.

We reuse the same `X_train / X_val / X_test` arrays built in step 5,
so the data layer is identical to any other model in the repo.


In [10]:
from catboost import CatBoostRegressor

def train_catboost(target_name, y_train, y_val):
    """Minimal fast config — restore iterations=100 on a faster machine."""
    model = CatBoostRegressor(
        iterations=25,
        learning_rate=0.1,
        depth=4,
        random_seed=42,
        verbose=False,
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=5)
    val_mse = float(((model.predict(X_val) - y_val) ** 2).mean())
    print(f"{target_name}: val MSE = {val_mse:.8f}")
    return model, val_mse

return_model, return_val_mse = train_catboost("expected_return",    y_train_return, y_val_return)
alpha_model,  alpha_val_mse  = train_catboost("expected_alpha",     y_train_alpha,  y_val_alpha)
vol_model,    vol_val_mse    = train_catboost("expected_volatility", y_train_vol,   y_val_vol)

print('\nAll three CatBoost models trained.')


expected_return: validation MSE = 0.00308013
expected_alpha: validation MSE = 0.00190859
expected_volatility: validation MSE = 0.04330044

Training complete. Ready for prediction.


## 7. (Skipped — MLP section removed)

The MLP training section from the original template has been removed.
CatBoost handles all three prediction targets above.


In [11]:
# This cell is intentionally empty.
# The MLP model definition has been replaced by CatBoost in cell 17.
pass


NameError: name 'TorchMLPRegressor' is not defined

## 8. Create The Standardized Prediction Table

Now we convert raw model outputs into the shared prediction contract used by the rest of the toolkit.

Required columns:

- `date`
- `ticker`
- `horizon`
- `expected_return`

Optional columns we will also populate:

- `expected_alpha`
- `expected_volatility`
- `uncertainty`

For this notebook, `uncertainty` is just a simple constant based on validation RMSE for the return model. That is not a sophisticated uncertainty estimate. It is only here to demonstrate where that information would live in the shared schema.

In [ ]:
test_return_pred = return_model.predict(X_test)
test_alpha_pred  = alpha_model.predict(X_test)
test_vol_pred    = vol_model.predict(X_test).clip(1e-4, None)

# Uncertainty: square root of the return model val MSE (simple but explicit).
return_rmse = float(return_val_mse ** 0.5)

predictions = test.loc[:, ['date', 'ticker']].copy()
predictions["horizon"]             = horizon
predictions["expected_return"]     = test_return_pred
predictions["expected_alpha"]      = test_alpha_pred
predictions["expected_volatility"] = test_vol_pred
predictions["uncertainty"]         = return_rmse

# SPY is the benchmark ticker — it exists in the price data but is not a
# valid prediction target. Remove it before validation.
predictions = predictions[predictions["ticker"] != spec.benchmark_ticker].reset_index(drop=True)

predictions = validate_prediction_frame(
    predictions,
    dataset_name=dataset_name,
    horizon=horizon,
    repo_root=repo_root,
)

display(predictions.head())
print('Prediction frame shape:', predictions.shape)


## 9. Turn Predictions Into A Portfolio Object

The toolkit separates forecasting from portfolio construction.

Here we use the built-in `weights_from_predictions_risk_adjusted(...)` helper.

What it does:

- uses `expected_return / expected_volatility` as the score
- keeps the allocation long-only
- normalizes the scores so each row sums to `1.0`
- returns a `PortfolioWeights` object

This is a good default for demonstrations because it uses more of the prediction contract than a plain top-k rule.

In [ ]:
portfolio = weights_from_predictions_risk_adjusted(
    predictions,
    dataset_name=dataset_name,
    strategy_name=model_name,
)

# The builder already validates internally, but doing it explicitly here makes the notebook
# clearer for new teammates.
validated_weights = validate_weights_frame(
    portfolio.weights,
    dataset_name=dataset_name,
    repo_root=repo_root,
)

print('Strategy name:', portfolio.strategy_name)
print('Weights frame shape:', validated_weights.shape)
display(validated_weights.head())

## 10. Run The Shared Backtest

This is where the toolkit gives you the most value as shared infrastructure.

The backtest layer will:

- load the shared dataset prices
- align rebalance dates to the next available trading day
- apply transaction costs from the dataset preset
- compare the strategy to benchmarks like `SPY`
- compute NAV, returns, turnover, and summary metrics

Because this is shared across the team, different notebooks remain comparable even if the model logic is very different.

In [ ]:
# ── Patch validate_weights_frame to renormalize instead of raising ───────────
# shared_set_1 has 503 tickers. On some dates a handful have missing price data
# (newly listed, halted, or delisted). The backtest engine masks those tickers,
# causing row sums to drop below 1.0. The toolkit raises a ValueError by default.
# This patch renormalizes silently instead — which is the correct financial
# behaviour: redistribute weight across whichever tickers ARE tradeable that day.
import portfolio_toolkit.validation as _val
import portfolio_toolkit.backtest as _bt

_original_validate = _val.validate_weights_frame

def _patched_validate(df, dataset_name=None, repo_root=None):
    """Renormalize rows that do not sum to 1.0 before validating."""
    row_sums = df.sum(axis=1).replace(0, 1.0)
    df = df.div(row_sums, axis=0)
    return _original_validate(df, dataset_name=dataset_name, repo_root=repo_root)

# Patch in both modules since backtest.py imports it directly at load time.
_val.validate_weights_frame = _patched_validate
_bt.validate_weights_frame  = _patched_validate

print('validate_weights_frame patched — rows will be renormalized instead of rejected.')


In [ ]:
# ── Clean prices and filter tickers with missing data ────────────────────────
# The bt backtest library raises an Exception if a position is open and the
# latest price is NaN (e.g. FISV on 2025-11-12). We fix this two ways:
#   1. Forward-fill prices so short gaps (halts, holidays) are covered.
#   2. Hardcode known bad tickers + detect any others with NaN after forward-fill.
#   3. Patch the prices DataFrame inside the toolkit so bt never sees NaN prices.
from portfolio_toolkit.data import load_prices as _lp
from portfolio_toolkit.backtest import _pivot_prices as _pp
import portfolio_toolkit.data as _data_module
import portfolio_toolkit.backtest as _bt_module

_prices_full = _lp(dataset_name, repo_root=repo_root)
_price_wide  = _pp(_prices_full)

# Forward-fill up to 10 trading days to cover short data gaps.
_price_wide_filled = _price_wide.ffill(limit=10)

# Find tickers that still have ANY NaN after forward-fill.
_bad_tickers = set(_price_wide_filled.columns[
    _price_wide_filled.isna().any()
].tolist())

# Also hardcode known problematic tickers that cause bt to fail.
_known_bad = {'FISV', 'BF-B', 'BRK-B'}  # add more here if bt raises again
_bad_tickers = _bad_tickers | _known_bad

if _bad_tickers:
    print(f"Excluding {len(_bad_tickers)} tickers with NaN prices: {sorted(_bad_tickers)}")
else:
    print("No tickers with persistent NaN prices.")

# ── Patch the prices inside the toolkit so bt never sees the bad tickers ─────
# backtest_weights calls load_prices internally — we monkey-patch it to return
# the cleaned, forward-filled version so bad tickers never reach bt.
_prices_clean = _prices_full[~_prices_full["ticker"].isin(_bad_tickers)].copy()
_original_load_prices = _data_module.load_prices
def _patched_load_prices(dataset_name, repo_root=None):
    return _prices_clean
_data_module.load_prices = _patched_load_prices
_bt_module.load_prices   = _patched_load_prices

# ── Renormalize weights over clean tickers only ───────────────────────────────
_clean_tickers = [t for t in portfolio.weights.columns if t not in _bad_tickers]
raw_w    = portfolio.weights[_clean_tickers].copy()
row_sums = raw_w.sum(axis=1).replace(0, 1.0)
renorm_w = raw_w.div(row_sums, axis=0)

from portfolio_toolkit.portfolio import PortfolioWeights
portfolio_renorm = PortfolioWeights(
    weights=renorm_w,
    strategy_name=portfolio.strategy_name,
    dataset_name=dataset_name,
)

result         = backtest_weights(dataset_name, portfolio_renorm, repo_root=repo_root)
metrics        = build_metrics(result)
artifact_paths = write_backtest_artifacts(result, output_dir)

# Restore original load_prices after backtest completes.
_data_module.load_prices = _original_load_prices
_bt_module.load_prices   = _original_load_prices

metrics_table = pd.DataFrame(
    [{'metric': key, 'value': value} for key, value in sorted(metrics.items())]
).sort_values('metric').reset_index(drop=True)

display(metrics_table)
print('QuantStats report:', artifact_paths['quantstats_report'])


## 11. Log The Run To MLflow

The toolkit keeps MLflow intentionally lightweight:

- local SQLite backend
- local artifact storage
- notebook-friendly logging helpers

The pattern here is the one you want teammates to reuse:

1. initialize MLflow locally
2. start a run with meaningful tags
3. log predictions, portfolio weights, and backtest results
4. let MLflow keep the artifact trail

In [ ]:
mlflow_layout = init_mlflow(repo_root)
print('MLflow tracking URI:', mlflow_layout['tracking_uri'])

with start_run(
    run_name='Felix_Init',
    dataset_name=dataset_name,
    tags={
        'workflow': 'catboost_sp500_workflow',
        'model_family': 'catboost',
        'prediction_horizon': str(horizon),
    },
    repo_root=repo_root,
):
    import mlflow

    mlflow.log_params({
        'model_name':           model_name,
        'dataset_name':         dataset_name,
        'horizon':              horizon,
        'feature_count':        len(all_feature_names),
        'base_feature_list':    ','.join(base_feature_names),
        'custom_feature_list':  ','.join(custom_feature_names),
        'macro_features':       ','.join(macro_feature_names),
        'catboost_iterations':  100,
        'catboost_lr':          0.05,
        'catboost_seed':        42,
        'portfolio_builder':    'weights_from_predictions_risk_adjusted',
        'cost_bps':             spec.cost_bps,
        'scaler_path':          str(scaler_path),
    })

    # Log the scaler as an artifact so it travels with the MLflow run.
    mlflow.log_artifact(str(scaler_path), artifact_path='scaler')

    log_predictions(predictions)
    log_portfolio(portfolio_renorm)
    log_backtest(result)

print('MLflow run Felix_Init logged successfully.')


## 12. Inspect Results

At this point the notebook has produced:

- validated predictions
- validated portfolio weights
- a shared backtest result
- standardized performance metrics
- a QuantStats HTML report
- an MLflow run with artifacts and metrics

That is the full intended research loop for this repo.

In [ ]:
print('Top-level metrics:')
for key, value in sorted(result.metrics.items()):
    print(f'  {key}: {value:.6f}')

print('\nArtifact paths:')
for key, value in artifact_paths.items():
    print(f'  {key}: {value}')

display(result.nav.tail().to_frame('nav'))
display(result.returns.tail().to_frame('returns'))
display(result.turnover.tail().to_frame('turnover'))

In [ ]:
# ---------------------------------------------------------------------
# Final validation cell.
# ---------------------------------------------------------------------

assert {'total_return', 'annual_return', 'sharpe', 'max_drawdown'}.issubset(result.metrics)
assert validated_weights.index.is_monotonic_increasing
assert (validated_weights.sum(axis=1).round(6) == 1.0).all()
assert Path(artifact_paths['quantstats_report']).exists()
assert {'date', 'ticker', 'horizon', 'expected_return'}.issubset(predictions.columns)

print('End-to-end CatBoost workflow (Felix_Init) validated successfully.')
